# Module 3: Embeddings & Representation Basics

## Learning Objectives
- Map tokens to dense vector representations
- Compute cosine similarity for semantic matching
- Implement nearest-neighbor retrieval
- Visualize embeddings with dimensionality reduction
- Connect to Kyro's `LlamaModel::embed_tokens`

## Concept: From Tokens to Vectors

An embedding layer creates a learned mapping from discrete token IDs to continuous vectors:

$$\text{embed}(w_i) = \mathbf{e}_i \in \mathbb{R}^d$$

During training, embeddings capture semantic relationships — similar words end up close in vector space. This is the foundation of how LLMs understand language.

**Kyro mapping**: `src/model/llama.rs::LlamaModel::new` creates `embed_tokens` on the first pipeline stage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Create a meaningful embedding space (not random)
np.random.seed(42)
vocab = ['king', 'queen', 'man', 'woman', 'apple', 'banana', 'fruit', 'food']
vocab_size = len(vocab)
dim = 8

# Create embeddings where semantically related words are close
embeddings = np.array([
    [0.9, 0.1, 0.8, 0.2, 0.1, 0.1, 0.1, 0.1],  # king
    [0.9, 0.1, 0.1, 0.8, 0.1, 0.1, 0.1, 0.1],  # queen
    [0.8, 0.2, 0.7, 0.3, 0.1, 0.1, 0.1, 0.1],  # man
    [0.8, 0.2, 0.1, 0.7, 0.1, 0.1, 0.1, 0.1],  # woman
    [0.1, 0.1, 0.1, 0.1, 0.9, 0.8, 0.7, 0.2],  # apple
    [0.1, 0.1, 0.1, 0.1, 0.8, 0.9, 0.8, 0.3],  # banana
    [0.1, 0.1, 0.1, 0.1, 0.8, 0.7, 0.9, 0.3],  # fruit
    [0.1, 0.1, 0.1, 0.1, 0.7, 0.8, 0.8, 0.9],  # food
])

print(f'Embedding matrix shape: {embeddings.shape}')
print(f'Vocabulary: {vocab}')

## Step 1: Cosine Similarity

In [ ]:
# Cosine similarity between embeddings
sim_matrix = cosine_similarity(embeddings)

print('Cosine Similarity Matrix:')
print(f'{"":<10}', end='')
for w in vocab:
    print(f'{w:<10}', end='')
print()
for i, w in enumerate(vocab):
    print(f'{w:<10}', end='')
    for j in range(len(vocab)):
        print(f'{sim_matrix[i][j]:.3f}     ', end='')
    print()

# Most similar pairs
print('\nMost similar pairs:')
for i in range(len(vocab)):
    for j in range(i+1, len(vocab)):
        if sim_matrix[i][j] > 0.95:
            print(f'  {vocab[i]} <-> {vocab[j]}: {sim_matrix[i][j]:.4f}')

## Step 2: Nearest-Neighbor Retrieval

In [ ]:
def nearest_neighbor(query_idx, embeddings, vocab, k=3):
    query = embeddings[query_idx].reshape(1, -1)
    sims = cosine_similarity(query, embeddings)[0]
    indices = np.argsort(sims)[::-1][1:k+1]  # Skip self
    return [(vocab[i], sims[i]) for i in indices]

# Query: find nearest neighbors to 'king'
query_idx = 0
neighbors = nearest_neighbor(query_idx, embeddings, vocab)
print(f'Nearest neighbors to "{vocab[query_idx]}":')
for word, sim in neighbors:
    print(f'  {word}: {sim:.4f}')

# Query: find nearest neighbors to 'apple'
query_idx = 4
neighbors = nearest_neighbor(query_idx, embeddings, vocab)
print(f'\nNearest neighbors to "{vocab[query_idx]}":')
for word, sim in neighbors:
    print(f'  {word}: {sim:.4f}')

## Step 3: Visualize with t-SNE

In [ ]:
# Dimensionality reduction for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
embeddings_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
colors = ['#e74c3c']*4 + ['#3498db']*4  # Royalty in red, fruit in blue
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=colors, s=100)
for i, word in enumerate(vocab):
    plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=12, ha='center', va='bottom')
plt.title('t-SNE Visualization of Token Embeddings')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.savefig('embeddings_tsne.png', dpi=100)
plt.show()
print('Saved embeddings_tsne.png')

## Step 4: RAG Sketch (Retrieval-Augmented Generation)

In [ ]:
# Simple RAG simulation
documents = {
    0: 'The cat sat on the mat',
    1: 'The dog ran in the park',
    2: 'The bird flew over the tree',
}

# Simulate document embeddings
doc_embeddings = np.random.randn(len(documents), dim)

def rag_query(query_text, query_embedding, doc_embeddings, docs, top_k=2):
    sims = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]
    results = [(docs[i], sims[i]) for i in top_indices]
    return results

query = np.array([0.85, 0.15, 0.75, 0.25, 0.1, 0.1, 0.1, 0.1])  # Like 'king'
results = rag_query('king-like query', query, doc_embeddings, documents)
print('RAG Results:')
for doc, score in results:
    print(f'  Score: {score:.4f} | Document: "{doc}"')

## Exercises
1. Use real pre-trained word embeddings (GloVe) and observe the semantic structure.
2. Implement a full RAG pipeline with a retriever and generator.
3. Compare Kyro's embedding initialization with pre-trained embeddings.
4. Explore `src/model/llama.rs::LlamaModel::embed_tokens` and the pipeline context.